# Loan Default Prediction — Research Prototype

**AIMLCZG546 — Software Engineering for Machine Learning · Assignment II · Group 40**

---

## What this notebook is, and why it exists

This is the **research code** half of Objective 1.2 (*"Clearly distinguish and demonstrate
the difference between research code vs production code"*). The production half is the
`src/loan_default/` package.

This notebook is the exploratory workflow that produced Assignment I's
`loan_default_pipeline.py`: load the file, look at it, find out what is wrong with it,
try three models, pick one. It is written the way exploratory work actually gets
written — top-to-bottom, `print()` for output, constants edited in place, nothing tested.
That is not a criticism of it. Research code has a different job: the goal is to *learn
something about the data as fast as possible*, and formality slows that down. The problem
is that research code does not stop being research code when it starts serving real
traffic.

**Provenance (stated honestly).** The modelling cells below are taken from Assignment I's
`loan_default_pipeline.py` essentially verbatim — same constants, same functions, same
`print()` formatting, same random seed. The *exploration* cells in section 2 are a
reconstruction: Assignment I submitted a finished script, and a script does not preserve
the investigation that led to its `LEAKAGE_COLS` list. Those cells re-derive the two
leakage findings from the actual dataset, so every number shown is computed here rather
than quoted from memory.

The final section compares the two codebases trait by trait.

## 1. Setup and configuration

Everything is a module-level constant, edited in the cell when it needs to change.
There is no configuration file and no way to run this with different settings without
editing the source.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

TARGET = "Status"
ID_AND_SENSITIVE = ["ID", "Gender"]  # identifier + protected attribute
RANDOM_STATE = 42
TEST_SIZE = 0.25

DATA_PATH = "../data/raw/loan_default_dataset.csv"

## 2. Load and look at the data

The first thing to establish: how big is it, and how imbalanced is the target?

In [2]:
print(f"[LOAD] Reading {DATA_PATH}")
df = pd.read_csv(DATA_PATH)
print(f"[LOAD] Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

balance = df[TARGET].value_counts(normalize=True).sort_index()
print(
    f"[LOAD] Target balance -> "
    f"0 (non-default): {balance.get(0, 0):.1%} | "
    f"1 (default): {balance.get(1, 0):.1%}"
)

[LOAD] Reading ../data/raw/loan_default_dataset.csv


[LOAD] Shape: 148,670 rows x 34 columns
[LOAD] Target balance -> 0 (non-default): 75.4% | 1 (default): 24.6%


About one loan in four defaults. That already rules out accuracy as the headline metric —
a model that predicts "never defaults" for everyone scores about 75% and is worthless.
PR-AUC is the metric to select on.

Note the target encoding is an **inference**: the dataset ships no data dictionary, so
"`Status = 1` means default" is the conventional reading, not a documented fact.

In [3]:
df.head()

,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


In [4]:
print("Column dtypes:")
print(df.dtypes.to_string())

Column dtypes:
ID                             int64
year                           int64
loan_limit                       str
Gender                           str
approv_in_adv                    str
loan_type                        str
loan_purpose                     str
Credit_Worthiness                str
open_credit                      str
business_or_commercial           str
loan_amount                    int64
rate_of_interest             float64
Interest_rate_spread         float64
Upfront_charges              float64
term                         float64
Neg_ammortization                str
interest_only                    str
lump_sum_payment                 str
property_value               float64
construction_type                str
occupancy_type                   str
Secured_by                       str
total_units                      str
income                       float64
credit_type                      str
Credit_Score                   int64
co-applicant_credit_typ

### 2.1 Missing values

Standard next step — which columns have holes in them?

In [5]:
missing = df.isna().mean().sort_values(ascending=False)
print("Missing fraction per column (non-zero only):")
print(missing[missing > 0].round(4).to_string())

Missing fraction per column (non-zero only):
Upfront_charges              0.2666
Interest_rate_spread         0.2464
rate_of_interest             0.2451
dtir1                        0.1622
property_value               0.1016
LTV                          0.1016
income                       0.0615
loan_limit                   0.0225
approv_in_adv                0.0061
age                          0.0013
submission_of_application    0.0013
loan_purpose                 0.0009
Neg_ammortization            0.0008
term                         0.0003


Three columns stand out at 24–27%: `Upfront_charges`, `Interest_rate_spread`,
`rate_of_interest`.

That number is suspiciously close to the 24.6% default rate. Coincidence, or not?

### 2.2 Leakage finding (a) — missingness leakage

The question to ask is not "how much is missing" but **"who is missing"**. If a column's
NaN pattern lines up with the target, the hole *is* the label.

In [6]:
print(
    f"{'column':24s} {'missing %':>10s} {'P(default | missing)':>22s} "
    f"{'P(default | present)':>22s}"
)
print("-" * 82)
for col in df.columns:
    if col == TARGET:
        continue
    is_missing = df[col].isna()
    if is_missing.sum() == 0:
        continue
    p_missing = df[TARGET][is_missing].mean()
    p_present = df[TARGET][~is_missing].mean()
    print(f"{col:24s} {is_missing.mean():>9.2%} {p_missing:>22.4f} {p_present:>22.4f}")

column                    missing %   P(default | missing)   P(default | present)
----------------------------------------------------------------------------------
loan_limit                   2.25%                 0.2635                 0.2461


approv_in_adv                0.61%                 0.2654                 0.2463
loan_purpose                 0.09%                 0.2612                 0.2464


rate_of_interest            24.51%                 1.0000                 0.0018
Interest_rate_spread        24.64%                 1.0000                 0.0000
Upfront_charges             26.66%                 0.9204                 0.0014
term                         0.03%                 0.3659                 0.2464
Neg_ammortization            0.08%                 0.2645                 0.2464
property_value              10.16%                 0.9999                 0.1613
income                       6.15%                 0.1354                 0.2537
age                          0.13%                 1.0000                 0.2454
submission_of_application     0.13%                 1.0000                 0.2454
LTV                         10.16%                 0.9999                 0.1613
dtir1                       16.22%                 0.6762                 0.1632


There it is. `Interest_rate_spread` and `rate_of_interest` are missing for rows that
default **100% of the time**. Let me confirm that exactly rather than trusting a rounded
number.

In [7]:
is_missing = df["Interest_rate_spread"].isna()
n_missing = int(is_missing.sum())
n_defaults = int(df[TARGET].sum())

print(f"Interest_rate_spread missing in : {n_missing:,} rows")
print(f"Total defaults in the dataset   : {n_defaults:,} rows")
print(f"Every missing row is a default  : {bool(df[TARGET][is_missing].eq(1).all())}")
print(f"The two sets are identical      : {n_missing == n_defaults}")

Interest_rate_spread missing in : 36,639 rows
Total defaults in the dataset   : 36,639 rows
Every missing row is a default  : True
The two sets are identical      : True


Exactly identical — 36,639 rows either way. This is not a weak correlation; the
missingness of `Interest_rate_spread` **is** the target, re-encoded.

The likely explanation is that these fields are populated when a loan is priced, and the
defaulted records in this extract never got that far. Whatever the cause, a model given
these columns would learn "no interest rate recorded → default", score beautifully in
testing, and be useless in production, where every live application has a rate.

**Decision: drop `Interest_rate_spread`, `rate_of_interest`, `Upfront_charges`.**
The third is included because 92% of its missing rows are defaults — the same mechanism,
slightly less absolute.

### 2.3 Leakage finding (b) — category leakage

Missingness is one way a column can encode the answer. A category value is another.

In [8]:
for col in df.select_dtypes(exclude=np.number).columns:
    grouped = df.groupby(col)[TARGET].agg(["mean", "size"])
    extreme = grouped[(grouped["mean"] > 0.9) | (grouped["mean"] < 0.02)]
    extreme = extreme[extreme["size"] > 500]
    if len(extreme):
        print(f"--- {col} ---")
        print(extreme.round(4).to_string())
        print()

--- credit_type ---
               mean   size
credit_type               
EQUI         0.9999  15298



In [9]:
print("Default rate by credit bureau:")
print(df.groupby("credit_type")[TARGET].agg(["mean", "size"]).round(4).to_string())

Default rate by credit bureau:
               mean   size
credit_type               
CIB          0.1580  48152
CRIF         0.1623  43901
EQUI         0.9999  15298
EXP          0.1599  41319


Three bureaus sit at a ~16% default rate. `EQUI` sits at **99.99% across 15,298 rows**.

A credit bureau does not cause default. Something about how `EQUI` records entered this
dataset is bound up with the outcome, so the column partly encodes the target.

> **Correction to Assignment I.** The Assignment I script's comment states that *all*
> 15,298 EQUI rows default. Recomputed here, it is 15,297 of 15,298 — one row is a
> non-default. The finding stands; the wording "100%" was slightly too strong. This is
> exactly the kind of quietly-wrong claim that survives in a notebook because nothing
> re-checks it.

In [10]:
equi = df[df["credit_type"] == "EQUI"]
print(f"EQUI rows            : {len(equi):,}")
print(f"EQUI defaults        : {int(equi[TARGET].sum()):,}")
print(f"EQUI non-defaults    : {int((equi[TARGET] == 0).sum()):,}")
print(f"EQUI default rate    : {equi[TARGET].mean():.6f}")

EQUI rows            : 15,298
EQUI defaults        : 15,297
EQUI non-defaults    : 1
EQUI default rate    : 0.999935


**Decision: drop `credit_type` entirely.** Dropping only the `EQUI` level would leave
the leak in place through the "not-EQUI" indicator, and the remaining bureaus carry
little signal anyway.

In [11]:
# Constants edited in place once the exploration above settled the question.
LEAKAGE_COLS = [
    "Interest_rate_spread",
    "rate_of_interest",
    "Upfront_charges",  # (a) missingness
    "credit_type",  # (b) category
]
print("Leakage columns to drop:", LEAKAGE_COLS)

Leakage columns to drop: ['Interest_rate_spread', 'rate_of_interest', 'Upfront_charges', 'credit_type']


## 3. Preprocessing — the Pipe-and-Filter pattern

Each stage is a "filter" that transforms the data and passes it on:

    raw -> impute missing -> encode categoricals -> scale numerics -> model

Note how the numeric/categorical split is decided: `select_dtypes` infers it **at runtime,
from whatever happens to be in this DataFrame**. Convenient while exploring. A liability
in production — see section 6.

In [12]:
def split_features_target(df):
    drop = [c for c in (ID_AND_SENSITIVE + LEAKAGE_COLS) if c in df.columns]
    print(f"[PREP] Dropping columns: {drop}")
    df = df.drop(columns=drop)

    y = df[TARGET].astype(int)
    X = df.drop(columns=[TARGET])

    num_cols = X.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()
    print(f"[PREP] {len(num_cols)} numeric, {len(cat_cols)} categorical features")
    return X, y, num_cols, cat_cols


def build_preprocessor(num_cols, cat_cols):
    # Numeric filter chain: impute median -> standardize
    numeric_filter = Pipeline(
        [
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]
    )
    # Categorical filter chain: impute most-frequent -> one-hot encode
    categorical_filter = Pipeline(
        [
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    return ColumnTransformer(
        [
            ("num", numeric_filter, num_cols),
            ("cat", categorical_filter, cat_cols),
        ]
    )


X, y, num_cols, cat_cols = split_features_target(df)

[PREP] Dropping columns: ['ID', 'Gender', 'Interest_rate_spread', 'rate_of_interest', 'Upfront_charges', 'credit_type']
[PREP] 8 numeric, 19 categorical features


In [13]:
print("Numeric features   :", num_cols)
print()
print("Categorical features:", cat_cols)

Numeric features   : ['year', 'loan_amount', 'term', 'property_value', 'income', 'Credit_Score', 'LTV', 'dtir1']

Categorical features: ['loan_limit', 'approv_in_adv', 'loan_type', 'loan_purpose', 'Credit_Worthiness', 'open_credit', 'business_or_commercial', 'Neg_ammortization', 'interest_only', 'lump_sum_payment', 'construction_type', 'occupancy_type', 'Secured_by', 'total_units', 'co-applicant_credit_type', 'age', 'submission_of_application', 'Region', 'Security_Type']


Worth pausing on this. `select_dtypes` put `year` in the numeric list because it happens
to be stored as an integer, and it put `total_units` (`"1U"`) and `age` (`"25-34"`) in
the categorical list because they happen to be strings. All three are right — **by
accident of how this particular CSV was written**, not because anything checked.

## 4. Model selection

Three candidates, one stratified 25% hold-out, selected on PR-AUC.

In [14]:
def candidate_models():
    # class_weight balanced addresses the ~75/25 imbalance.
    return {
        "Logistic Regression": LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        "Hist Gradient Boosting": HistGradientBoostingClassifier(
            class_weight="balanced", random_state=RANDOM_STATE
        ),
    }


def evaluate(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    return {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
    }

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f"[SPLIT] train {X_train.shape[0]:,} | test {X_test.shape[0]:,} (stratified)")

pre = build_preprocessor(num_cols, cat_cols)
results, fitted = [], {}

for name, clf in candidate_models().items():
    print(f"[TRAIN] {name} ...")
    pipe = Pipeline([("preprocess", pre), ("model", clf)])
    pipe.fit(X_train, y_train)
    results.append(evaluate(name, pipe, X_test, y_test))
    fitted[name] = pipe

results_df = pd.DataFrame(results).set_index("model")

[SPLIT] train 111,502 | test 37,168 (stratified)
[TRAIN] Logistic Regression ...


[TRAIN] Random Forest ...


[TRAIN] Hist Gradient Boosting ...


## 5. Results

In [16]:
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

print("=" * 70)
print("MODEL COMPARISON  (test set)")
print("=" * 70)
print(results_df.to_string())

best_name = results_df["pr_auc"].idxmax()
print(f"\n[SELECT] Best model by PR-AUC: {best_name}")

MODEL COMPARISON  (test set)
                        accuracy  precision  recall     f1  roc_auc  pr_auc
model                                                                      
Logistic Regression       0.6684     0.3943  0.6443 0.4892   0.7235  0.5008
Random Forest             0.8887     0.9345  0.5898 0.7232   0.8889  0.8309
Hist Gradient Boosting    0.8737     0.7517  0.7281 0.7397   0.8945  0.8418

[SELECT] Best model by PR-AUC: Hist Gradient Boosting


In [17]:
best = fitted[best_name]
y_pred = best.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(f"Confusion matrix for {best_name}:")
print("                 pred:0   pred:1")
print(f"   actual 0    {cm[0, 0]:>8d} {cm[0, 1]:>8d}")
print(f"   actual 1    {cm[1, 0]:>8d} {cm[1, 1]:>8d}")

Confusion matrix for Hist Gradient Boosting:
                 pred:0   pred:1
   actual 0       25805     2203
   actual 1        2491     6669


In [18]:
def print_top_features(pipe, name, k=12):
    """Show the most influential features (interpretability quality req)."""
    feat_names = pipe.named_steps["preprocess"].get_feature_names_out()
    model = pipe.named_steps["model"]
    if hasattr(model, "feature_importances_"):
        vals, label = model.feature_importances_, "importance"
    elif hasattr(model, "coef_"):
        vals, label = np.abs(model.coef_[0]), "|coefficient|"
    else:
        print("\n[FEATURES] Model exposes no importances/coefficients.")
        return
    order = np.argsort(vals)[::-1][:k]
    print(f"\nTop {k} features by {label} ({name}):")
    for i in order:
        print(f"   {feat_names[i]:40s} {vals[i]:.4f}")


print_top_features(fitted["Logistic Regression"], "Logistic Regression")
print_top_features(fitted["Random Forest"], "Random Forest")


Top 12 features by |coefficient| (Logistic Regression):
   cat__lump_sum_payment_lpsm               1.4819
   cat__lump_sum_payment_not_lpsm           1.0314
   cat__Neg_ammortization_neg_amm           0.7415
   cat__construction_type_mh                0.7105
   cat__Security_Type_Indriect              0.7105
   cat__Secured_by_land                     0.7105
   cat__co-applicant_credit_type_EXP        0.6313
   cat__total_units_3U                      0.5768
   cat__submission_of_application_to_inst   0.5484
   cat__loan_limit_ncf                      0.5353
   cat__loan_purpose_p2                     0.5016
   cat__open_credit_nopc                    0.4363



Top 12 features by importance (Random Forest):
   num__LTV                                 0.1765
   num__property_value                      0.1575
   num__dtir1                               0.1392
   num__income                              0.0942
   num__Credit_Score                        0.0742
   num__loan_amount                         0.0696
   cat__co-applicant_credit_type_EXP        0.0178
   cat__co-applicant_credit_type_CIB        0.0160
   num__term                                0.0152
   cat__Neg_ammortization_neg_amm           0.0121
   cat__Neg_ammortization_not_neg           0.0117
   cat__lump_sum_payment_not_lpsm           0.0115


The selected model is Hist Gradient Boosting on PR-AUC. Logistic Regression collapses to
roughly 0.50 PR-AUC once the leaky columns are gone — which is itself informative: in an
earlier run *with* `credit_type` included it scored around 0.77, and that gap is a direct
measurement of how much it had been leaning on the leak.

**These numbers are what this configuration produces** — this seed, this single split,
these preprocessing choices, this assumed target direction. They are not "the" answer.

In [19]:
import joblib

out = "loan_default_model.joblib"
joblib.dump(best, out)
print(f"[SAVE] Selected model '{best_name}' saved to {out}")

[SAVE] Selected model 'Hist Gradient Boosting' saved to loan_default_model.joblib


---

# 6. Research code vs production code

Everything above **works**. It loads real data, finds real problems, trains real models,
and produces a real artifact. So what is missing?

The gap is not correctness. It is that this notebook answers *"does this idea work?"*,
while the `src/loan_default/` package has to answer *"will this keep working, unattended,
on data nobody has looked at, when the person who wrote it has moved on?"*

## 6.1 Trait by trait

| # | Research code (this notebook) | Production code (`src/loan_default/`) | Why the change matters |
|---|---|---|---|
| 1 | Column types inferred by `select_dtypes` at runtime | `NUMERIC_FEATURES` / `CATEGORICAL_FEATURES` pinned in `data.py` | A serving payload of one row can infer types differently from a 148k-row file. Same code, different features, no error. |
| 2 | `print()` to stdout | `logging_config.get_logger()` with INFO / WARNING / ERROR | `print` vanishes under a process manager. Levels let you page on ERROR and ignore INFO. |
| 3 | Failures surface as raw tracebacks | `SchemaValidationError`, `DataQualityError`, `FileNotFoundError`, HTTP 503 | A typed error tells a caller *what* went wrong and whether retrying helps. |
| 4 | Constants edited in the source | `config/config.yaml` | Change a threshold without editing (or redeploying) code. |
| 5 | No tests | 88 pytest tests | Nothing here would notice if a refactor silently swapped two columns. |
| 6 | Leakage found by hand, once | `DataQualityChecker` — 8 automated checks | A one-off human check does not protect next month's data. |
| 7 | Runs top to bottom, in order, or not at all | Importable modules with single responsibilities | You cannot call cell 12 of a notebook from an API request handler. |
| 8 | Artifact written to the working directory | `CONFIG["model"]["artifact_path"]`, resolved absolutely | A relative path means the code only works from one directory. |
| 9 | Model reachable only by re-running the notebook | FastAPI `/predict` with a Pydantic schema | Other systems need an interface, not a copy of your environment. |
| 10 | No input validation — any DataFrame is accepted | Pydantic request model, 422 on malformed input | Production input is hostile or careless, not curated. |
| 11 | Fairness/leakage exclusions are comments | `test_invariance_to_excluded_columns` asserts them | A comment cannot fail the build. A test can. |
| 12 | State lives in the kernel; cells can run out of order | Functions with explicit arguments and return values | Out-of-order execution is a bug class that simply cannot exist in the package. |

## 6.2 The point that generalises

Traits 6 and 11 are the ones specific to *ML* systems rather than software generally.

Ordinary software fails loudly — a null pointer, a 500, a crash. ML systems fail
**silently**. Give this model a scrambled feature matrix and it returns a confident
probability. Feed it next quarter's data after an upstream schema change and it returns
confident probabilities. Nothing crashes. The output stays the same *shape*, and only the
*meaning* is wrong.

That is why the production version invests where a normal refactor would not bother: an
automated data-quality gate, invariance tests, directional tests, drift monitoring. They
exist to make silent failures loud.

## 6.3 What the production version does *not* improve

Being fair to the notebook:

- **The model is identical.** Same three candidates, same seed, same PR-AUC. Production
  engineering did not make it more accurate.
- **The exploration had to happen this way.** Sections 2.2 and 2.3 are what a notebook is
  *for*. Discovering the missingness leak took several pivots — running that as a test
  suite would have been slower and worse.
- **A notebook is better for communicating a finding.** The missingness table above is
  more persuasive read top-to-bottom than the same logic as `check_missingness_leakage()`.

The lesson is not "notebooks are bad". It is that the notebook is the **right tool for
the first question and the wrong tool for the second**, and that the transition between
them is real engineering work rather than a tidy-up.